### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, ExperimentDataPreprocessor, set_seed
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (0/0):
Data count before: 480608
Data count after: 480608
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480608
Num of distinct users: 2103
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480608
interaction data count after merging: 478564
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358129 (74.83%
valid: 46916 (9.8%)
test: 73519 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555886
1    0.444114
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585155
1    0.414845
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358129
  Num of positive interactions: 159050 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 159050], edge_label=[159050])
Edge Index: tensor([[   0,    0,    0,  ..., 2093, 2093, 2093],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head()

Calculating user diversity preference scale:   0%|          | 0/2094 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2094/2094 [00:33<00:00, 63.17it/s]


,userID,actorID_dist,actorID_dps,country_dist,country_dps,directorID_dist,directorID_dps,genre_dist,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545
1,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.475977,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.300297,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.383176,"[15.0, 27.0, 12.0, 3.5, 30.0, 7.0, 0.0, 54.5, ...",0.764622
2,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 0.0, 0.0, ...",0.485033,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.098869,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.384691,"[26.0, 11.0, 0.0, 3.0, 34.5, 6.5, 1.0, 22.5, 1...",0.793320
3,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.562356,"[0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, ...",0.111710,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.470833,"[55.5, 42.5, 9.0, 9.5, 91.5, 39.5, 0.0, 64.5, ...",0.814879
4,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.518630,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.297905,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.429368,"[44.5, 46.0, 0.0, 0.0, 35.0, 24.0, 0.0, 110.0,...",0.756129


In [9]:
encoded_train_df_with_dps = encoded_train_df.merge(user_dps_df, on="userID", how="left")

#### Prepare prediction pool for inference/testing

In [10]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2095
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2095(users) * 500(items) = 1047500


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1047495,2094,7957,0,"[5403, 102, 5844, 14185, 8714]",63,955,"[8, 15, 0, 0, 0, 0, 0, 0]"
1047496,2094,1611,0,"[13198, 4752, 15371, 5573, 8531]",63,2910,"[4, 5, 0, 0, 0, 0, 0, 0]"
1047497,2094,4715,0,"[2653, 2800, 5578, 9181, 11244]",63,1393,"[7, 18, 0, 0, 0, 0, 0, 0]"
1047498,2094,1924,0,"[3069, 7855, 11716, 8412, 5873]",62,2293,"[5, 0, 0, 0, 0, 0, 0, 0]"
1047499,2094,1590,0,"[6359, 5740, 13573, 3459, 14058]",63,2954,"[1, 2, 8, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [11]:
# TODO: determine which Dataset to use
from common.datasets import UserItemPairDataset
from torch.utils.data import DataLoader

BATCH_SIZE = 1024

train_dataset = UserItemPairDataset(encoded_train_df_with_dps)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358129
valid data count: 46916
test data count: 1047500


### Configure Model (LightningModule)

In [12]:
from models.mtdp_gcn_cf_rec import MTDPRecGCNCF

EMB_DIM = 32
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}
MT_WEIGHTS = {
    "rec_loss": 1.0,
    "dps_loss": 0.5,
}

model = MTDPRecGCNCF(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
    dps_weights=DPS_WEIGHTS,
    mt_weights=MT_WEIGHTS,
)


### Configure Trainer and Experiment

In [13]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "mtdp-gcn-bce-v1-exp"
RUN_NAME = "test3.2"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(EXPERIMENT_NAME, RUN_NAME, PATIENCE)

In [14]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    deterministic=True,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [15]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/experiments/test_checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type            | Params | Mode 
-------------------------------------------------------
0 | gcn_model  | GraphConvModule | 376 K  | train
1 | dps_module | DPSPredictor    | 132    | train
-------------------------------------------------------
377 K     Trainable params
0         Non-trainable params
377 K     Total params
1.508     Total estimated model params size (MB)
48        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 12, global step 4550: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 5 records. Best score: 0.544. Signaling Trainer to stop.
Epoch 13, global step 4900: 'val_f1' was not in top 1


🏃 View run test3.2 at: http://140.112.106.216:3683/#/experiments/4/runs/d2aa84136909498cb8d65469679bf433
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/4


### Inference

In [15]:
# NOTE: the inference model MUST be the same as the training model
best_model_path = "test_checkpoints/mtdp-gcn-bce-v1-exp-test3.2-best-checkpoint-epoch=08-val_f1=0.54.ckpt"

model = MTDPRecGCNCF.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
    dps_weights=DPS_WEIGHTS,
    mt_weights=MT_WEIGHTS,
)


In [16]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.6711322069168091     │
│          test_f1          │    0.12000051140785217    │
│         test_loss         │      1131.232421875       │
│         test_prec         │    0.06506983935832977    │
│         test_rec          │    0.7701236009597778     │
└───────────────────────────┴───────────────────────────┘

🏃 View run test3.2 at: http://140.112.106.216:3683/#/experiments/4/runs/c570baaa6c544f4cb557029a2a8d9c75
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/4


[{'test_loss': 1131.232421875,
  'test_acc': 0.6711322069168091,
  'test_prec': 0.06506983935832977,
  'test_rec': 0.7701236009597778,
  'test_f1': 0.12000051140785217}]

In [17]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2094, 2094, 2094]),
 'item': tensor([7977, 4839,  979,  ..., 4715, 1924, 1590]),
 'score': tensor([ 2.5437e+03,  1.4225e+02,  2.8831e+03,  ..., -1.4756e-01,
         -5.2400e-01, -9.8381e-01]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'metric': {'test_loss': 1131.232421875,
  'test_acc': 0.6711322195704057,
  'test_prec': 0.0650698403727775,
  'test_rec': 0.770123610610184,
  'test_f1': 0.12000051090135772},
 'user_emb': tensor([[-1.0556e+01, -4.5465e+00, -6.0520e+00,  ..., -5.4699e+00,
          -1.0001e+01, -6.9859e+00],
         [-5.1466e+00, -2.2324e+00, -2.7581e+00,  ..., -2.6160e+00,
          -4.6799e+00, -3.4097e+00],
         [-9.2370e-01, -3.8943e-01, -4.9873e-01,  ..., -4.7047e-01,
          -8.3145e-01, -6.0587e-01],
         ...,
         [-2.9484e+01, -1.2596e+01, -1.6722e+01,  ..., -1.5213e+01,
          -2.7620e+01, -1.9387e+01],
         [-5.3881e+01, -2.3291e+01, -2.9686e+01,  ..., -2.7590e+01,
          -4.9953e+01, 

In [18]:
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)
eval_df

,user,rec_items,gt_items
0,75,"[2959, 318, 2571, 4993, 5952, 7153, 356, 1252,...","[45722, 1233, 110, 2959, 2571]"
1,78,"[46578, 1732, 1221, 44694, 52281, 1225, 3160, ...","[4119, 6993, 8400, 50872]"
2,127,"[2959, 4878, 48780, 923, 4235, 1225, 1175, 172...","[45726, 6958]"
3,170,"[296, 50, 48394, 44191, 1136, 1222, 3949, 260,...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[48394, 5902, 40819, 5995, 7147, 6502, 6, 5154...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."
...,...,...,...
2090,71509,"[750, 6711, 2329, 1258, 1617, 5902, 39292, 399...","[2019, 2238, 6783, 1231, 8914, 53887, 2010, 10..."
2091,71525,"[2329, 6874, 7438, 4011, 4027, 589, 7147, 1653...","[49530, 47099, 49278, 7147, 51575, 48304]"
2092,71529,"[1089, 5952, 32587, 1198, 7438, 3996, 110, 391...","[3996, 5952, 786, 1917, 2355, 1682]"
2093,71534,"[7361, 2858, 1136, 778, 6874, 7438, 33166, 923...","[1227, 40819, 7361, 4235, 2692, 2927, 1266, 12..."


In [21]:
eval_score_df = evaluator.evaluate(eval_df, K=5)
eval_score_df = evaluator.evaluate(eval_score_df, K=10)
eval_score_df = evaluator.evaluate(eval_score_df, K=20)
eval_score_df.describe()

,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000,2095.000000
mean,0.384505,0.100134,0.197804,0.436188,0.179660,0.188926,0.466356,0.300426,0.170334
std,0.378704,0.157677,0.223575,0.326964,0.212982,0.181450,0.276608,0.258014,0.145989
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.295846,0.105965,0.050000
50%,0.430677,0.040000,0.200000,0.455605,0.111111,0.100000,0.480388,0.250000,0.150000
75%,0.679731,0.142857,0.400000,0.682966,0.250000,0.300000,0.683425,0.437500,0.250000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000
